# Mean Reversion Experiment

Testing whether extreme 3-day returns tend to revert in the next day.

**Hypothesis**: Large moves (positive or negative) over 3 days should be followed by opposite moves in the next day (mean reversion).

In [ ]:
# Setup imports - now works automatically after pip install -e .
from pathlib import Path
import pandas as pd

from mini_research_lab import (
    download_prices,
    add_return_features,
    MiniResearchLab,
    default_experiments,
    plot_experiment_bundle,
)

In [26]:
# Download and prepare data
prices = download_prices("AAPL", start="2018-01-01")
df = add_return_features(prices)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

Data shape: (2087, 14)
Date range: 2018-01-02 00:00:00 to 2026-04-22 00:00:00


,close,high,low,open,volume,ret_1d,ret_3d,ret_5d,abs_ret_1d,fwd_ret_1d,fwd_ret_3d,fwd_abs_ret_1d,ma_20,dist_from_ma20
date,,,,,,,,,,,,,,
2018-01-02,40.304180,40.313541,39.602261,39.812839,102223600,NaN,NaN,NaN,NaN,-0.000174,0.015906,0.000174,NaN,NaN
2018-01-03,40.297153,40.839972,40.233983,40.367346,118071600,-0.000174,NaN,NaN,0.000174,0.004645,0.012309,0.004645,NaN,NaN
2018-01-04,40.484329,40.587278,40.262056,40.369681,89738400,0.004645,NaN,NaN,0.004645,0.011386,0.007513,0.011386,NaN,NaN
2018-01-05,40.945271,41.031839,40.489024,40.580273,94640000,0.011386,0.015906,NaN,0.011386,-0.003714,-0.004058,0.003714,NaN,NaN
2018-01-08,40.793190,41.087995,40.694918,40.793190,82271200,-0.003714,0.012309,NaN,0.003714,-0.000115,0.005334,0.000115,NaN,NaN


In [27]:
# Set up experiment
lab = MiniResearchLab(df)
spec = default_experiments()[0]  # mean_reversion_3d_to_1d

print(f"Experiment: {spec.name}")
print(f"Description: {spec.description}")
print(f"X variable: {spec.x_col}")
print(f"Y variable: {spec.y_col}")

Experiment: mean_reversion_3d_to_1d
Description: Mean reversion mini-project using recent 3-day return vs next 1-day return.
X variable: ret_3d
Y variable: fwd_ret_1d


In [28]:
# Run experiment
result = lab.run_experiment(spec.x_col, spec.y_col)

# Display results
print("=== X Variable Summary ===")
print(pd.Series(result["x_describe"].to_dict()))

print("\n=== Y Variable Summary ===")
print(pd.Series(result["y_describe"].to_dict()))

=== X Variable Summary ===
series                           ret_3d
count                              2084
mean                           0.003257
std                             0.03197
min                           -0.189513
25%                           -0.013878
50%                            0.004246
75%                            0.021449
max                            0.151848
iqr                            0.035327
lower_fence                   -0.066869
upper_fence                     0.07444
n_lower_outliers                     52
n_upper_outliers                     34
skew_hint           left-skew candidate
dtype: object

=== Y Variable Summary ===
series                       fwd_ret_1d
count                              2086
mean                           0.001103
std                             0.01928
min                           -0.128647
25%                            -0.00792
50%                            0.001165
75%                            0.010903
max        

In [29]:
# Regression results
print("=== Regression Summary ===")
regression_df = pd.DataFrame([result["regression"].to_dict()]).T
regression_df.columns = ['value']
print(regression_df)

print("\n=== Full Model Summary ===")
print(result["model"].summary())

=== Regression Summary ===
                value
x_col          ret_3d
y_col      fwd_ret_1d
intercept    0.001186
coef        -0.027449
std_err      0.013212
t_value     -2.077651
p_value      0.037864
r_squared     0.00207
n_obs            2083
ci_low      -0.053359
ci_high      -0.00154

=== Full Model Summary ===
                            OLS Regression Results                            
Dep. Variable:             fwd_ret_1d   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     4.317
Date:                Wed, 22 Apr 2026   Prob (F-statistic):             0.0379
Time:                        20:17:20   Log-Likelihood:                 5270.8
No. Observations:                2083   AIC:                        -1.054e+04
Df Residuals:                    2081   BIC:                        -1.053e+04
Df Model:                           1            

In [30]:
# Generate and save plots to shared reports/figures
saved_files = plot_experiment_bundle(
    df, spec.x_col, spec.y_col, 
    title_prefix=spec.name,
    output_dir="../reports/figures"  # Explicit path to shared reports
)

print("Saved plots:")
for file_path in saved_files:
    print(f"  {file_path}")

Saved plots:
  ../reports/figures/mean_reversion_3d_to_1d — ret_3d_histogram.png
  ../reports/figures/mean_reversion_3d_to_1d — ret_3d_boxplot.png
  ../reports/figures/mean_reversion_3d_to_1d — fwd_ret_1d_histogram.png
  ../reports/figures/mean_reversion_3d_to_1d — fwd_ret_1d_boxplot.png
  ../reports/figures/mean_reversion_3d_to_1d — ret_3d_vs_fwd_ret_1d_scatter.png


In [31]:
# Save tables to shared reports/tables
tables_dir = Path("../reports/tables")
tables_dir.mkdir(parents=True, exist_ok=True)

# Save summaries
x_summary = pd.DataFrame([result["x_describe"].to_dict()]).T
x_summary.to_csv(tables_dir / f"{spec.name}_x_summary.csv")

y_summary = pd.DataFrame([result["y_describe"].to_dict()]).T
y_summary.to_csv(tables_dir / f"{spec.name}_y_summary.csv")

reg_summary = pd.DataFrame([result["regression"].to_dict()]).T
reg_summary.to_csv(tables_dir / f"{spec.name}_regression.csv")

with open(tables_dir / f"{spec.name}_model_summary.txt", "w") as f:
    f.write(str(result["model"].summary()))

print(f"Tables saved to {tables_dir}")

Tables saved to ../reports/tables


## Interpretation

### Key Results:
- **Coefficient**: Shows the relationship between 3-day returns and next-day returns
- **P-value**: Indicates statistical significance (< 0.05 is significant)
- **R-squared**: How much variance is explained (usually low for single predictors)

### What this means:
- A negative coefficient suggests mean reversion (big moves reverse)
- A positive coefficient suggests momentum (big moves continue)
- Low R-squared is normal - financial returns are hard to predict

### Distribution insights:
- Check skewness hints in the summaries
- Look at outlier counts (they can dominate results)
- Examine the histograms to see if assumptions hold